# Lesson 4, one call at a time

Lesson 4 turned lesson 3's line into a classifier, logistic regression, and spent two
thirds of its time on how to judge one: the confusion matrix, precision and recall, the
threshold, the ROC curve and its area, the threshold chosen from costs, class weights,
and three classes. This notebook takes each step on three to twenty drives, small
enough to do with a pencil, and recomputes the worked examples of lesson 4's commentary
(`Docs/classification_and_metrics_commentary.md`) with SciPy and scikit-learn, so that
the arithmetic by hand and the library call sit side by side.

**How to use it.** Each section is one *atom*. Before each code cell that teaches
something there is a question under *Your prediction, before you run the next cell*:
answer it first, then run. Notebook `00_the_scikit_learn_api.ipynb` explains `fit`,
`predict`, `predict_proba` and the threshold; its atoms 9 and 10 (`decision_function`,
and the thresholds that `roc_curve` and `precision_recall_curve` return) belong with
this lesson.

The notation is lesson 4's: $p$ the predicted probability of failing,
$z = w^\top x + b$ the log-odds, $\sigma$ the sigmoid, $y = 1$ for a drive that failed,
TP, FP, FN, TN the true and false positives and negatives, $t^*$ the cost-optimal
threshold.

Not examinable. The full-scale version of each atom is in lesson 4's notebooks
`01_logistic_regression_from_scratch.ipynb`, `02_confusion_matrix_and_metrics.ipynb`
and `03_roc_thresholds_and_imbalance.ipynb`.

## Setup

In [1]:
# Everything used here ships with the course image:
# pip install numpy==1.26.4 scipy scikit-learn==1.3.1

import numpy as np
from scipy.special import expit, logit, softmax
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, auc, average_precision_score,
                             classification_report, confusion_matrix, f1_score,
                             log_loss, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

np.set_printoptions(precision=4, suppress=True)

## 1. Odds, log-odds, and the sigmoid that undoes them

The **odds** of an event are $p / (1 - p)$; the **log-odds** are their natural
logarithm. SciPy names the two directions: `logit(p)` gives the log-odds and
`expit(z)` the sigmoid, $1 / (1 + e^{-z})$. Notebook 1 writes the sigmoid by hand and
never calls them.

*From:* lesson 4, slide *Why the log-odds*; commentary, worked example *five
probabilities, their odds and their log-odds*.

In [2]:
p = np.array([0.05, 0.2, 0.5, 0.8, 0.95])

### Your prediction, before you run the next cell

Which two of the five probabilities will have log-odds of the same size and
opposite sign? And what will `expit` give back when it is applied to the log-odds?

In [3]:
odds = p / (1 - p)
z = logit(p)
for pi, oi, zi in zip(p, odds, z):
    print(f"p = {pi:<5}  odds = {pi:.2f} / {1 - pi:.2f} = {oi:6.3f}   log-odds = {zi:+.2f}")
print("\nlogit(p) equals log(odds):", np.allclose(z, np.log(odds)))
print("expit(logit(p)):", expit(z))

p = 0.05   odds = 0.05 / 0.95 =  0.053   log-odds = -2.94
p = 0.2    odds = 0.20 / 0.80 =  0.250   log-odds = -1.39
p = 0.5    odds = 0.50 / 0.50 =  1.000   log-odds = +0.00
p = 0.8    odds = 0.80 / 0.20 =  4.000   log-odds = +1.39
p = 0.95   odds = 0.95 / 0.05 = 19.000   log-odds = +2.94

logit(p) equals log(odds): True
expit(logit(p)): [0.05 0.2  0.5  0.8  0.95]


**What to look at.** 0.2 and 0.8 sit at −1.39 and +1.39, and 0.05 and 0.95 at −2.94
and +2.94: the log-odds are symmetric around 0, which is where $p = 0.5$ lands. The
odds are not symmetric (0.25 against 4) and they stop at 0. `expit` returns the five
probabilities exactly: the sigmoid is the inverse of the log-odds, which is the whole
content of slide *Log-odds of p = P(y = 1 | x): a line*.

**Try changing.** add 0.999 and 0.001 to `p`: how far out do their log-odds go?

## 2. A coefficient multiplies the odds, not the probability

In logistic regression one unit more of a feature adds its coefficient $w$ to the
log-odds, so it multiplies the odds by $e^w$. What that does to the probability depends
on where the drive starts. `expit(logit(p) + w)` is the whole computation.

*From:* slide *What the coefficients mean*; commentary, worked example *one
coefficient, three drives*.

In [4]:
w = 0.7
p_before = np.array([0.01, 0.5, 0.9])

### Your prediction, before you run the next cell

With $w = 0.7$ the odds roughly double. For which of the three drives does the
probability roughly double too?

In [5]:
p_after = expit(logit(p_before) + w)
print(f"e^w = {np.exp(w):.2f}")
for pb, pa in zip(p_before, p_after):
    ob, oa = pb / (1 - pb), pa / (1 - pa)
    print(f"p {pb:<4} -> {pa:.3f}   odds {ob:6.3f} -> {oa:6.3f} (x {oa / ob:.2f})"
          f"   probability x {pa / pb:.2f}")

e^w = 2.01
p 0.01 -> 0.020   odds  0.010 ->  0.020 (x 2.01)   probability x 1.99
p 0.5  -> 0.668   odds  1.000 ->  2.014 (x 2.01)   probability x 1.34
p 0.9  -> 0.948   odds  9.000 -> 18.124 (x 2.01)   probability x 1.05


**What to look at.** $e^{0.7} = 2.01$, and every drive's odds are multiplied by
exactly that. The probability nearly doubles only for the rare event, 0.01 to 0.020; at
0.5 it rises by a third, to 0.668; at 0.9 it moves five points, to 0.948. "Twice the
odds" and "twice as likely" agree only when the event is rare, which is why the
confusion survives on the disk fleet, where failures are rare, and fails elsewhere.

**Try changing.** use `w = -0.7`: does the drive at 0.9 lose as much probability as the drive at 0.5?

## 3. `log_loss`: the model's average surprise

The cost logistic regression minimises is the log loss: for each drive, minus the log
of the probability the model gave to what actually happened, averaged over the drives.
`log_loss(y, p)` takes the labels and the probabilities of the positive class.

*From:* slides *Surprise: minus the log of a probability* and *Cross-entropy, also
called log loss*; commentary, worked example *four drives, one confident mistake*.

In [6]:
y = np.array([1, 0, 0, 1])                # drives A, B, C, D: 1 = failed
p = np.array([0.9, 0.2, 0.1, 0.05])       # each drive's predicted P(fail)

### Your prediction, before you run the next cell

Which drive will contribute the most to the cost, and will its loss be larger
or smaller than the other three together?

In [7]:
happened = np.where(y == 1, p, 1 - p)     # probability of what actually happened
per_drive = -np.log(happened)
for name, h, l in zip("ABCD", happened, per_drive):
    print(f"drive {name}: probability of what happened {h:.2f}, loss {l:.3f}")
print(f"\naverage by hand {per_drive.mean():.3f}   log_loss {log_loss(y, p):.3f}")

drive A: probability of what happened 0.90, loss 0.105
drive B: probability of what happened 0.80, loss 0.223
drive C: probability of what happened 0.90, loss 0.105
drive D: probability of what happened 0.05, loss 2.996

average by hand 0.857   log_loss 0.857


**What to look at.** Drive D failed after the model gave it 0.05: its loss, 2.996, is
larger than the other three together, 0.105 + 0.223 + 0.105. The average, 0.857, is
exactly what `log_loss` prints. A cost dominated by confident mistakes is what a fitting
procedure should be pushed by; notebook 1 computes the same quantity on 6,000 drives at
every iteration of its gradient descent.

**Try changing.** set D's probability to 0.001, then to 0.5: how does the cost move?

## 4. One step of gradient descent, by hand

Notebook 1 runs 4,000 iterations of the same update in a loop: the error $p - y$ for
every drive, then $w$ moves against the average of error × $x$, and $b$ against the
average error. Here is one iteration on three drives, with the gradient checked against
a numerical derivative of the cost.

*From:* slide *Lesson 3's gradient, $\hat{y} = \sigma(w^\top x + b)$*; commentary,
worked example *one step of gradient descent on three drives*.

In [8]:
x = np.array([2.0, -1.0, 0.5])            # one standardised feature
y = np.array([1, 0, 0])
w, b, alpha = 0.0, 0.0, 0.5

def cost(w, b):
    return log_loss(y, expit(w * x + b), labels=[0, 1])

### Your prediction, before you run the next cell

Starting from $w = 0$, does the first step move $w$ up or down? Which of the three
drives pulls it the other way?

In [9]:
p = expit(w * x + b)
error = p - y
grad_w, grad_b = (error * x).mean(), error.mean()
h = 1e-6
numerical = (cost(w + h, b) - cost(w - h, b)) / (2 * h)
print("p:", p, "  error:", error, "  error * x:", error * x)
print(f"dJ/dw = {grad_w:.4f} (numerically {numerical:.4f})   dJ/db = {grad_b:.4f}")
w_new, b_new = w - alpha * grad_w, b - alpha * grad_b
print(f"after one step: w = {w_new:.4f}, b = {b_new:.4f}")
print(f"cost {cost(w, b):.4f} -> {cost(w_new, b_new):.4f}")

p: [0.5 0.5 0.5]   error: [-0.5  0.5  0.5]   error * x: [-1.   -0.5   0.25]
dJ/dw = -0.4167 (numerically -0.4167)   dJ/db = 0.1667
after one step: w = 0.2083, b = -0.0833
cost 0.6931 -> 0.6006


**What to look at.** Every drive starts at $p = 0.5$. The gradient for $w$ is −0.4167,
and the numerical derivative of the cost agrees, so the step raises $w$ to 0.2083; $b$
falls to −0.0833 because two of the three drives are healthy. The failed drive at
$x = 2$ pulls $w$ up, and so does the healthy drive at $x = -1$ (a larger $w$ lowers its
$p$); only the healthy drive at $x = 0.5$ pulls $w$ down, and weakly. The cost falls from
0.6931, the log loss of a coin flip, to 0.6006.

**Try changing.** set `alpha = 20`: does the cost still fall after one step?

## 5. `confusion_matrix(...).ravel()`: four numbers in a fixed order

For two classes, `confusion_matrix(y_true, y_pred)` has truth in rows and prediction in
columns, and `.ravel()` flattens it row by row: **TN, FP, FN, TP**, in that order.
Notebooks 2 and 3 unpack it as `tn, fp, fn, tp = ...ravel()` without stopping on why.
Every metric of the lesson comes from the four counts. scikit-learn has no
`specificity_score`, but specificity is the recall of the negative class:
`recall_score(..., pos_label=0)`.

*From:* slides *Four numbers instead of one* and *Our model, in those terms*;
commentary, worked example *twenty drives, four of which fail*.

In [10]:
# 20 test drives: the first 4 failed, the other 16 stayed healthy
y_true = np.array([1, 1, 1, 1] + [0] * 16)
y_pred = np.array([1, 1, 1, 0] + [1, 1] + [0] * 14)   # 3 caught, 1 missed, 2 false alarms

### Your prediction, before you run the next cell

What will `.ravel()` print first, the true negatives or the true positives? And
what is F1, from $2TP / (2TP + FP + FN)$?

In [11]:
cm = confusion_matrix(y_true, y_pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print("ravel():", cm.ravel(), " ->  TN", tn, " FP", fp, " FN", fn, " TP", tp)

dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((20, 1)), y_true)
print(f"\naccuracy     {accuracy_score(y_true, y_pred):.3f}   always healthy "
      f"{dummy.score(np.zeros((20, 1)), y_true):.3f}")
print(f"precision    {precision_score(y_true, y_pred):.3f}   = {tp} / {tp + fp}")
print(f"recall       {recall_score(y_true, y_pred):.3f}   = {tp} / {tp + fn}")
print(f"F1           {f1_score(y_true, y_pred):.3f}   = {2 * tp} / {2 * tp + fp + fn}")
print(f"specificity  {recall_score(y_true, y_pred, pos_label=0):.3f}   = {tn} / {tn + fp}")

[[14  2]
 [ 1  3]]
ravel(): [14  2  1  3]  ->  TN 14  FP 2  FN 1  TP 3

accuracy     0.850   always healthy 0.800
precision    0.600   = 3 / 5
recall       0.750   = 3 / 4
F1           0.667   = 6 / 9
specificity  0.875   = 14 / 16


**What to look at.** `.ravel()` gives 14, 2, 1, 3: TN first and TP last, the reading
order of the matrix. Accuracy 0.85 against 0.80 for the always-healthy model; precision
3 of 5 = 0.60, recall 3 of 4 = 0.75, F1 6 of 9 = 0.667, specificity 14 of 16 = 0.875,
so a false positive rate of 0.125. Unpacking in another order, `tp, fp, fn, tn = ...`,
would swap TP and TN silently: no error, every metric wrong.

**Try changing.** flag one more healthy drive (turn one of the 14 trailing zeros of `y_pred` into a 1): which metrics move, and which stay put?

## 6. `classification_report`: the two averages, by hand

`classification_report` prints one row per class and then two averages. **macro avg**
is the plain mean over the classes; **weighted avg** weights each class by its
**support**, the number of drives whose *true* class it is. Notebook 2 prints the
report; here are its last two rows recomputed.

*From:* slide *Read the minority class, not the average*.

### Your prediction, before you run the next cell

On the same twenty drives, will the weighted-average F1 be closer to the healthy
class's F1 or to the failed class's?

In [12]:
print(classification_report(y_true, y_pred, digits=3))
f1_per_class = f1_score(y_true, y_pred, average=None)     # [healthy, failed]
support = np.bincount(y_true)                              # [16, 4]
macro = f1_per_class.mean()
weighted = (f1_per_class * support).sum() / support.sum()
print(f"F1 per class {f1_per_class.round(3)}, support {support}")
print(f"macro    = ({f1_per_class[0]:.3f} + {f1_per_class[1]:.3f}) / 2 = {macro:.3f}"
      f"   f1_score(average='macro')    {f1_score(y_true, y_pred, average='macro'):.3f}")
print(f"weighted = (16 x {f1_per_class[0]:.3f} + 4 x {f1_per_class[1]:.3f}) / 20 = {weighted:.3f}"
      f"   f1_score(average='weighted') {f1_score(y_true, y_pred, average='weighted'):.3f}")

              precision    recall  f1-score   support

           0      0.933     0.875     0.903        16
           1      0.600     0.750     0.667         4

    accuracy                          0.850        20
   macro avg      0.767     0.812     0.785        20
weighted avg      0.867     0.850     0.856        20

F1 per class [0.903 0.667], support [16  4]
macro    = (0.903 + 0.667) / 2 = 0.785   f1_score(average='macro')    0.785
weighted = (16 x 0.903 + 4 x 0.667) / 20 = 0.856   f1_score(average='weighted') 0.856


**What to look at.** The healthy class's F1 is 0.903 (with healthy as the positive
class, 2 × 14 ÷ (2 × 14 + 1 + 2) = 28 ÷ 31), the failed class's 0.667. Macro gives the
two the same say: 0.785. Weighted gives the 16 healthy drives four times the say of the
4 failed ones: 0.856, close to the healthy class and to the accuracy, 0.850. On the disk
fleet the gap is much wider, 0.820 against 0.975, because there the healthy drives are
96% of the test set instead of 80%. Which to report is the application's choice; the
minority class's own row is the one that must stay visible.

**Try changing.** make the test set 4 failures and 96 healthy drives, all correctly left alone (append 80 zeros to both arrays): what happens to the two averages?

## 7. `roc_auc_score`: ties count half, and only the order matters

Lesson 2's atom 02·12 counted the pairs behind an AUC. Lesson 4 adds two details. A tie
between a failing and a healthy drive counts half a pair. And anything that keeps the
order of the scores keeps the AUC, while every decision taken at a threshold can change.

*From:* slides *What the area means* and *What AUC does not tell you*; commentary,
worked example *the AUC of seven drives, by counting pairs*.

In [13]:
scores = np.array([0.9, 0.6, 0.3, 0.7, 0.3, 0.2, 0.1])
y = np.array([1, 1, 1, 0, 0, 0, 0])        # three failing drives, four healthy

### Your prediction, before you run the next cell

Counting the tie at 0.3 as half a pair, what is the AUC? And after dividing every
score by 10?

In [14]:
failing, healthy = scores[y == 1], scores[y == 0]
pairs = sum(1.0 if f > h else 0.5 if f == h else 0.0 for f in failing for h in healthy)
print(f"pairs won: {pairs} of {len(failing) * len(healthy)} = {pairs / 12:.3f}")
print(f"roc_auc_score: {roc_auc_score(y, scores):.3f}")
fpr, tpr, _ = roc_curve(y, scores)
print(f"area under roc_curve's points: {auc(fpr, tpr):.3f}")

scaled = scores / 10
print(f"\ndivided by 10: AUC {roc_auc_score(y, scaled):.3f}")
print("flagged at 0.5 before:", (scores >= 0.5).sum(), "  after:", (scaled >= 0.5).sum())

pairs won: 9.5 of 12 = 0.792
roc_auc_score: 0.792
area under roc_curve's points: 0.792

divided by 10: AUC 0.792
flagged at 0.5 before: 3   after: 0


**What to look at.** 9.5 of the 12 pairs: 0.792, the same from counting, from
`roc_auc_score` and from the area under the points `roc_curve` returns. Divided by 10,
the AUC is 0.792 again, but at threshold 0.5 three drives were flagged before and none
after. The AUC sees only the order, so it cannot tell whether a probability of 0.7
means anything: that is calibration, a separate property.

**Try changing.** give the healthy drive at 0.3 a score of 0.31: by how much does the AUC fall?

## 8. `average_precision_score`: a step sum, not a trapezoid

`precision_recall_curve` returns the precision and recall at every threshold, and
**average precision (AP)** adds each threshold's precision weighted by the recall it
adds: $\text{AP} = \sum_k (R_k - R_{k-1}) P_k$. A trapezoid drawn through the same
points gives a different number, and is not what `average_precision_score` computes.

*From:* slide *The rule: report both*; commentary, worked example *average precision of
a seven-drive ranking*.

In [15]:
# seven drives sorted riskiest first: failed, healthy, failed, healthy, healthy, failed, healthy
y = np.array([1, 0, 1, 0, 0, 1, 0])
scores = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3])

### Your prediction, before you run the next cell

What is the AP of this ranking? And what would a ranking drawn at random score:
0.5, or something else?

In [16]:
precision, recall, thresholds = precision_recall_curve(y, scores)
print("thresholds:", thresholds)
print("precision: ", precision.round(3))
print("recall:    ", recall.round(3))
step_sum = -np.sum(np.diff(recall) * precision[:-1])
print(f"\nstep sum {step_sum:.3f}   average_precision_score {average_precision_score(y, scores):.3f}")
print(f"trapezoid through the same points {auc(recall, precision):.3f}")
print(f"a random ranking: precision = positive rate = {y.mean():.3f} at every recall")

thresholds: [0.3 0.4 0.5 0.6 0.7 0.8 0.9]
precision:  [0.429 0.5   0.4   0.5   0.667 0.5   1.    1.   ]
recall:     [1.    1.    0.667 0.667 0.667 0.333 0.333 0.   ]

step sum 0.722   average_precision_score 0.722
trapezoid through the same points 0.678
a random ranking: precision = positive rate = 0.429 at every recall


**What to look at.** `precision_recall_curve` returns one point per threshold plus a final
point with recall 0 and precision 1, so `precision` and `recall` have one entry more than
`thresholds`, and the arrays run from the lowest threshold to the highest. Recall rises
only at the three cuts that pass a failed drive, where precision is 1.000, 0.667 and
0.500: each weighted by its recall step of 1/3, they give 0.722, from the step sum and
from `average_precision_score` alike. The trapezoid through the same points gives
0.678, a different number. A random ranking scores the positive rate,
3 ÷ 7 = 0.429: that, not 0.5, is the line to beat.

**Try changing.** swap the first two drives (healthy first, then failed): what happens to the AP, and to the AUC of the same ranking?

## 9. A threshold from two costs, applied to the probabilities

`predict` always compares the probability with 0.5. To decide with another threshold,
take column 1 of `predict_proba` and compare it yourself. The cost-optimal threshold is
$t^* = C_{FP} / (C_{FP} + C_{FN})$, the probability at which flagging and leaving a
drive cost the same in expectation.

*From:* slides *Where the two costs cross* and *Where they cross, in one line*;
commentary, worked example *invented costs: a false alarm €50, a missed failure €200*.

In [17]:
c_fp, c_fn = 50, 200
p = np.array([0.15, 0.3])                 # two drives' predicted P(fail)

### Your prediction, before you run the next cell

With a false alarm at 50 and a miss at 200, is the drive at 0.15 flagged? And the
one at 0.3? What would `predict`'s 0.5 do with them?

In [18]:
t_star = c_fp / (c_fp + c_fn)
print(f"t* = {c_fp} / ({c_fp} + {c_fn}) = {t_star:.2f}")
for pi in p:
    flag, leave = (1 - pi) * c_fp, pi * c_fn
    print(f"p = {pi}: flagging costs {flag:.1f}, leaving {leave:.1f} -> "
          f"{'flag' if flag < leave else 'leave'};  p > t*: {pi > t_star};  p >= 0.5: {pi >= 0.5}")

t* = 50 / (50 + 200) = 0.20
p = 0.15: flagging costs 42.5, leaving 30.0 -> leave;  p > t*: False;  p >= 0.5: False
p = 0.3: flagging costs 35.0, leaving 60.0 -> flag;  p > t*: True;  p >= 0.5: False


**What to look at.** $t^* = 0.20$. The drive at 0.15 costs 42.5 to flag and 30.0 to
leave, in expectation, so it is left; the drive at 0.3 costs 35.0 against 60.0 and is
flagged. The rule $p > t^*$ gives the same two decisions, and `predict`'s 0.5 would flag
neither. With equal costs $t^*$ is 0.5: the default is right only then, and only if
the probabilities are calibrated.

**Try changing.** set `c_fn = 50`: what is $t^*$, and which drives are flagged now?

## 10. `class_weight`: the weights, and the threshold they stand for

`class_weight="balanced"` gives each class the weight $m / (2 \times \text{class size})$,
from the class frequencies, not from any cost; `compute_class_weight` prints it.
Weights in the ratio $k$ raise the intercept by about $\log k$, so the weighted model's
0.5 falls where the plain model's $1 / (1 + k)$ was. Notebook 3 computes the weights by
hand and checks the equivalence on the fleet.

*From:* slides *What class weights do* and *Class weights: the same lever, pulled
earlier*; commentary, worked example *a weight ratio of 4*.

In [19]:
labels = np.array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0])   # ten drives, two failures

### Your prediction, before you run the next cell

What weights does `"balanced"` give these ten drives, and what is their ratio?
What threshold does that ratio stand for?

In [20]:
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=labels)
k = weights[1] / weights[0]
print(f"weights: healthy {weights[0]:.3f} = 10 / (2 x 8), failed {weights[1]:.3f} = 10 / (2 x 2);"
      f" ratio k = {k:.0f}")
print(f"stands for a threshold of 1 / (1 + {k:.0f}) = {1 / (1 + k):.2f}")
odds = 0.2 / 0.8
print(f"a drive at p = 0.20: odds {odds:.2f}, times {k:.0f} = {odds * k:.0f}, "
      f"probability {expit(logit(0.2) + np.log(k)):.2f}")

weights: healthy 0.625 = 10 / (2 x 8), failed 2.500 = 10 / (2 x 2); ratio k = 4
stands for a threshold of 1 / (1 + 4) = 0.20
a drive at p = 0.20: odds 0.25, times 4 = 1, probability 0.50


The equivalence is exact only for a correctly specified model and a large sample. The
next cell generates 5,000 drives from a logistic model in one feature, so the model
*is* correctly specified, and fits it with and without the weights. `penalty=None` turns
off scikit-learn's default penalty, which would otherwise pull on the coefficients.

In [21]:
rng = np.random.default_rng(0)
x_sim = rng.normal(size=(5000, 1))
y_sim = (rng.random(5000) < expit(-3 + 1.5 * x_sim[:, 0])).astype(int)

### Your prediction, before you run the next cell

With weights `{0: 1, 1: 4}`, by how much will the fitted intercept rise? Will the
slope change much?

In [22]:
plain = LogisticRegression(penalty=None).fit(x_sim, y_sim)
weighted = LogisticRegression(penalty=None, class_weight={0: 1, 1: 4}).fit(x_sim, y_sim)
print(f"failures: {y_sim.sum()} of 5,000")
print(f"intercept: plain {plain.intercept_[0]:.3f}, weighted {weighted.intercept_[0]:.3f}, "
      f"shift {weighted.intercept_[0] - plain.intercept_[0]:.3f} against log 4 = {np.log(4):.3f}")
print(f"slope:     plain {plain.coef_[0, 0]:.3f}, weighted {weighted.coef_[0, 0]:.3f}")
same = ((plain.predict_proba(x_sim)[:, 1] >= 0.2) == (weighted.predict(x_sim) == 1)).mean()
print(f"plain at 0.2 and weighted at 0.5 take the same decision on {same:.1%} of the drives")

failures: 494 of 5,000
intercept: plain -2.994, weighted -1.603, shift 1.391 against log 4 = 1.386
slope:     plain 1.531, weighted 1.525
plain at 0.2 and weighted at 0.5 take the same decision on 100.0% of the drives


**What to look at.** `"balanced"` gives 0.625 to each healthy drive and 2.5 to each
failure, a ratio of 4, from the class frequencies alone (8 healthy against 2 failed): the
costs of the two errors play no part. A ratio of 4 stands for a threshold of
1 ÷ (1 + 4) = 0.20: a drive the plain model puts at 0.20 has odds 0.25, and multiplied by
4 they become 1, a probability of 0.50. On the simulated drives the intercept rises by
1.391, against log 4 = 1.386, the slope barely moves (1.531 against 1.525), and the two
models take the same decision on 100.0% of the drives. That is the correctly specified
case; with a misspecified model, few data or a strong penalty, the weights also move the
slope, and the match is only approximate.

**Try changing.** replace the generating rule by one the model cannot represent, for example `expit(-3 + 1.5 * x_sim[:, 0] ** 2)`, and fit again: does the slope still stay put?

## 11. Softmax, and `predict_proba` for three classes

With three classes a logistic regression computes one score per class,
$z_k = w_k^\top x + b_k$, and the **softmax** turns them into probabilities:
exponentiate each score, divide by their sum. `scipy.special.softmax` does it, and a
fitted model's `predict_proba` is the softmax of its `decision_function` scores.

*From:* slide *Three classes: softmax (optional)*; commentary, worked example *softmax on three
scores*.

In [23]:
scores = np.array([2.0, 1.0, 0.1])        # healthy, degraded, failed

### Your prediction, before you run the next cell

Which class wins, and with a probability above or below 0.5? With two classes and
scores 0 and 1.5, what does softmax give for the second class?

In [24]:
by_hand = np.exp(scores) / np.exp(scores).sum()
print("e^z:", np.exp(scores).round(3), " sum", round(np.exp(scores).sum(), 3))
print("by hand:", by_hand.round(3), "  softmax:", softmax(scores).round(3))
print(f"two classes, scores (0, 1.5): {softmax([0.0, 1.5])[1]:.3f}   sigmoid(1.5) = {expit(1.5):.3f}")

# a three-class model on nine drives with one reading each
X3 = np.array([[0.1], [0.4], [0.8], [1.6], [2.0], [2.3], [3.1], [3.5], [4.0]])
y3 = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])          # healthy, degraded, failed
model3 = LogisticRegression().fit(X3, y3)
z3 = model3.decision_function(X3[[1, 4, 7]])
print("\nscores of three drives:\n", z3.round(3))
print("softmax of the scores equals predict_proba:",
      np.allclose(softmax(z3, axis=1), model3.predict_proba(X3[[1, 4, 7]])))

e^z: [7.389 2.718 1.105]  sum 11.213
by hand: [0.659 0.242 0.099]   softmax: [0.659 0.242 0.099]
two classes, scores (0, 1.5): 0.818   sigmoid(1.5) = 0.818

scores of three drives:
 [[ 1.631  0.496 -2.127]
 [-0.291  0.503 -0.212]
 [-2.093  0.51   1.584]]
softmax of the scores equals predict_proba: True


**What to look at.** $e^2 = 7.389$, $e^1 = 2.718$, $e^{0.1} = 1.105$, sum 11.213: the
drive is called healthy with 0.659, degraded 0.242, failed 0.099, and the three add up to
1. The winner is below 0.5 only when the scores are close, which is why a three-class
decision uses the largest probability, not a 0.5 threshold. With two classes and scores
0 and 1.5, softmax gives 0.818, which is $\sigma(1.5)$: the sigmoid is softmax with one
score fixed at 0. The fitted model gives each drive three scores, one per class, and its
`predict_proba` is exactly their softmax; notebook 3 does the same on the three-class
fleet.

**Try changing.** add 10 to all three scores of the first example: do the probabilities change?